### 1. Cài đặt và Khởi tạo môi trường
Phần này thực hiện cài đặt thư viện cần thiết, kết nối Google Drive và thiết lập cấu trúc thư mục dự án.

In [ ]:
!pip install -q pdfplumber uuid

import os
import json
import re
import unicodedata
import pdfplumber
import uuid

# Kết nối Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/NLP"
except Exception as e:
    print("Môi trường không phải Colab. Sử dụng Local Path.")
    BASE_PATH = "."

# Cấu hình các thư mục chuẩn hóa
RAW_DIR = os.path.join(BASE_PATH, "data/raw")
PROCESSED_DIR = os.path.join(BASE_PATH, "data/processed")
DATASET_DIR = os.path.join(BASE_PATH, "data/datasets")

for path in [RAW_DIR, PROCESSED_DIR, DATASET_DIR]:
    os.makedirs(path, exist_ok=True)

PDF_INPUT = os.path.join(RAW_DIR, "23_2008_QH12_82203.pdf")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Trích xuất văn bản từ PDF
Sử dụng `pdfplumber` để đọc nội dung từ file PDF đầu vào và lưu lại bản thô.

In [ ]:
def extract_and_save_raw(pdf_path: str) -> str:
    print(f"[*] Đang trích xuất dữ liệu từ: {pdf_path}")
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                all_text += text + "\n"

    raw_txt_path = os.path.join(RAW_DIR, "raw_content.txt")
    with open(raw_txt_path, "w", encoding="utf-8") as f:
        f.write(all_text)
    print(f"[+] Đã lưu Raw Text ({len(all_text)} ký tự).")
    return all_text

### 3. Làm sạch và Chuẩn hóa văn bản
Chuẩn hóa Unicode, loại bỏ các ký tự nhiễu và định dạng lại cấu trúc các Điều/Khoản luật.

In [ ]:
def clean_and_save(text: str) -> str:
    print("[*] Đang làm sạch và chuẩn hóa văn bản...")
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'(?i)Luật này đã được Quốc hội.*?(Nguyễn Phú Trọng|CHỦ TỊCH QUỐC HỘI).*?$', '', text, flags=re.DOTALL)
    text = re.sub(r'(?i)(Trang\s+\d+(/\d+)?|Header_Text|Footer_Text)', '', text)
    text = re.sub(r'[^\w\s\.,;:\(\)\đ\Đ\-\"]', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'(?<![:\.])\n+', ' ', text)
    text = re.sub(r'(Điều\s+\d+\.)', r'\n\n\1', text)
    text = re.sub(r'(?m)^(\s*\d+\.\s)', r'\n\1', text)
    text = re.sub(r'(?m)^(\s*[a-zđ]\)\s)', r'\n\1', text)
    text = text.strip()

    clean_txt_path = os.path.join(PROCESSED_DIR, "clean_content.txt")
    with open(clean_txt_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[+] Đã lưu Clean Text ({len(text)} ký tự).")
    return text

### 4. Chia nhỏ văn bản (Chunking) và Gán Metadata
Chia văn bản thành các đơn vị nhỏ hơn (chunks) và trích xuất thông tin như mức phạt, phương tiện, hành vi vi phạm.

In [ ]:
def generate_parent_child_dataset(clean_text: str):
    print("[*] Đang thực hiện Parent-Child Chunking...")

    parents = []
    children = []

    # Tách văn bản thành các Parent Documents (Cấp Điều)
    articles = re.split(r'(?=\nĐiều\s+\d+\.)', '\n' + clean_text)

    for article in articles:
        article = article.strip()
        if not article: continue

        match = re.search(r'^(Điều\s+\d+)\.\s*(.*?)(?=\n|$)', article)
        if not match: continue

        article_num = match.group(1).strip()
        article_title = match.group(2).strip()

        # 1. TẠO PARENT DOCUMENT
        parent_id = str(uuid.uuid4()) # Tạo ID duy nhất cho Parent
        parent_content = f"{article_num}. {article_title}\n" + article[match.end():].strip()

        parents.append({
            "doc_id": parent_id,
            "content": parent_content,
            "metadata": {
                "source": "Luật Giao thông Đường bộ 2008",
                "article": article_num,
                "type": "parent_document"
            }
        })

        # 2. TẠO CHILD CHUNKS (Cấp Khoản)
        article_body = article[match.end():].strip()
        sub_sections = re.split(r'(?m)^(?=\d+\.\s)', article_body)

        for section in sub_sections:
            section = section.strip()
            if not section: continue

            # Phân loại logic Metadata cơ bản cho Child Chunks
            section_lower = section.lower()
            metadata = {
                "parent_id": parent_id,
                "article": article_num,
                "type": "child_chunk",
                "vehicle": None,
                "violation": None,
                "penalty": None
            }

            if article_num == "Điều 3" or re.search(r'\b(là|gồm|được hiểu là)\b', section_lower[:50]):
                metadata["type"] = "definition"
            elif "phạt tiền từ" in section_lower:
                metadata["type"] = "violation"
                penalty_match = re.search(r'(?i)phạt tiền từ\s+(.*?)\s+đến\s+(.*?)đồng', section)
                if penalty_match: metadata["penalty"] = f"Từ {penalty_match.group(1)} đến {penalty_match.group(2)} đồng"
                vehicle_match = re.search(r'(?i)(xe ô tô|xe máy|mô tô|xe gắn máy|xe đạp)', section)
                if vehicle_match: metadata["vehicle"] = vehicle_match.group(0).lower()
                violation_match = re.search(r'(?i)(điều khiển xe|thực hiện một trong các hành vi)(.*?)(?=phạt tiền|bị phạt|;$|\.$)', section)
                if violation_match: metadata["violation"] = violation_match.group(2).strip().strip(':,')

            children.append({
                "content": section,
                "metadata": metadata
            })

    return parents, children

### 5. Chạy Pipeline
Thực thi toàn bộ quy trình từ đầu đến cuối.

In [ ]:
if __name__ == "__main__":
    if not os.path.exists(PDF_INPUT):
        print(f"[!] Lỗi: Không tìm thấy file {PDF_INPUT}.")
    else:
        raw_content = extract_and_save_raw(PDF_INPUT)
        clean_content = clean_and_save(raw_content)
        parents, children = generate_parent_child_dataset(clean_content)

        parent_file = os.path.join(DATASET_DIR, "parents_traffic_law.json")
        child_file = os.path.join(DATASET_DIR, "children_traffic_law.json")

        with open(parent_file, "w", encoding="utf-8") as f:
            json.dump(parents, f, ensure_ascii=False, indent=4)

        with open(child_file, "w", encoding="utf-8") as f:
            json.dump(children, f, ensure_ascii=False, indent=4)

        print("="*60)
        print(f"Tổng số Parent Documents (Điều): {len(parents)}")
        print(f"Tổng số Child Chunks (Khoản): {len(children)}")

[*] Đang trích xuất dữ liệu từ: /content/drive/MyDrive/NLP/data/raw/23_2008_QH12_82203.pdf
[+] Đã lưu Raw Text (411128 ký tự).
[*] Đang làm sạch và chuẩn hóa văn bản...
[+] Đã lưu Clean Text (412320 ký tự).
[*] Đang thực hiện Parent-Child Chunking...
Tổng số Parent Documents (Điều): 176
Tổng số Child Chunks (Khoản): 707
